In [ ]:
!pip install -q openai sentence-transformers pypdf

In [ ]:
from google.colab import userdata
from openai import OpenAI

PROVEEDORES = {
    # nombre: (dirección de la API, nombre del secreto, modelo sugerido)
    "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai/", "GEMINI_API_KEY", "gemini-flash-lite-latest"),
    "groq": ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "llama-3.1-8b-instant"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "ESCRIBA_UN_MODELO:free"),
    "deepseek": ("https://api.deepseek.com", "DEEPSEEK_API_KEY", "deepseek-flash"),
}

PROVEEDOR = "gemini"   # cambie aquí: gemini, groq, openrouter o deepseek
url, secreto, MODELO = PROVEEDORES[PROVEEDOR]
cliente = OpenAI(base_url=url, api_key=userdata.get(secreto))

for m in cliente.models.list():
    print(m.id)

In [ ]:
respuesta = cliente.chat.completions.create(
    model=MODELO,
    messages=[{"role": "user", "content": "Explica en 3 líneas qué es un embedding."}],
)
print(respuesta.choices[0].message.content)
print(respuesta.usage)

In [ ]:
import json

pedido = "Clasifica este comentario de un cliente: 'La app se cierra cada vez que pago con PSE'."
respuesta = cliente.chat.completions.create(
    model=MODELO,
    messages=[
        {"role": "system", "content": "Responde solo con un JSON con las claves "
         "categoria (error, sugerencia o felicitacion), urgencia (1 a 5) y resumen."},
        {"role": "user", "content": pedido},
    ],
    response_format={"type": "json_object"},
)
datos = json.loads(respuesta.choices[0].message.content)
datos

In [ ]:
from pypdf import PdfReader

RUTA = "guia.pdf"
paginas = [p.extract_text() or "" for p in PdfReader(RUTA).pages]

fragmentos = []
for numero, texto in enumerate(paginas, start=1):
    for inicio in range(0, len(texto), 700):
        fragmento = texto[max(0, inicio - 100): inicio + 700]
        if fragmento.strip():
            fragmentos.append({"pagina": numero, "texto": fragmento})

print(len(fragmentos), "fragmentos")

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectores = embedder.encode([f["texto"] for f in fragmentos], normalize_embeddings=True)


def buscar(pregunta, k=4):
    v = embedder.encode([pregunta], normalize_embeddings=True)[0]
    similitud = vectores @ v
    mejores = np.argsort(similitud)[::-1][:k]
    return [fragmentos[i] | {"similitud": float(similitud[i])} for i in mejores]


for r in buscar("¿Cuántas horas dura el curso?"):
    print(r["pagina"], round(r["similitud"], 2), r["texto"][:80].replace("\n", " "))

In [ ]:
def responder(pregunta):
    contexto = "\n\n".join(f"[página {f['pagina']}]\n{f['texto']}" for f in buscar(pregunta))
    respuesta = cliente.chat.completions.create(
        model=MODELO,
        temperature=0,
        messages=[
            {"role": "system", "content":
                "Eres el asistente de la guía del curso. Responde en español usando SOLO el "
                "contexto. Indica la página de donde sale la respuesta, por ejemplo (página 3). "
                "Si la respuesta no está en el contexto, responde: 'No encuentro esa información en la guía'."},
            {"role": "user", "content": f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"},
        ],
    )
    return respuesta.choices[0].message.content


print(responder("¿Con qué imágenes se entrena el modelo de la semana 3?"))
print(responder("¿Cuál es la capital de Australia?"))